# R11 — полный OCR в Google Colab

Notebook клонирует ветку R11, распаковывает изображения на быстрый локальный диск Colab и сохраняет возобновляемый кеш на Google Drive. Перед запуском выберите GPU: `Runtime → Change runtime type → GPU`.

В `MyDrive/ecup/input/` должны лежать `images.zip` и `image_manifest.parquet`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/ecup')
INPUT_DIR = DRIVE_ROOT / 'input'
CACHE_DIR = DRIVE_ROOT / 'cache' / 'ocr'
OUTPUT_DIR = DRIVE_ROOT / 'output'
ARCHIVE_PATH = INPUT_DIR / 'images.zip'
MANIFEST_PATH = INPUT_DIR / 'image_manifest.parquet'
assert ARCHIVE_PATH.exists(), f'Не найден {ARCHIVE_PATH}'
assert MANIFEST_PATH.exists(), f'Не найден {MANIFEST_PATH}'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Google Drive готов')

In [ ]:
import subprocess
import torch
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'GPU не подключена в Colab'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
if GPU_GIB >= 35:
    MAX_PIXELS, MAX_NEW_TOKENS = 1_003_520, 512
elif GPU_GIB >= 20:
    MAX_PIXELS, MAX_NEW_TOKENS = 602_112, 384
else:
    MAX_PIXELS, MAX_NEW_TOKENS = 401_408, 256
print(GPU_NAME, f'{GPU_GIB:.1f} GiB')
print('OCR profile:', MAX_PIXELS, 'pixels,', MAX_NEW_TOKENS, 'tokens')

In [ ]:
import subprocess
REPO_URL = 'https://github.com/zimmer10/quality-control.git'
BRANCH = 'feature/R11-ocr-cache'
REPO_DIR = Path('/content/quality-control')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
print('Код готов:', REPO_DIR)

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[ocr]'], check=True)
print('Зависимости установлены')

In [ ]:
LOCAL_DATA = Path('/content/ecup_data')
IMAGES_ROOT = LOCAL_DATA / 'images'
EXTRACTED_MARKER = LOCAL_DATA / '.images_complete'
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
if not EXTRACTED_MARKER.exists():
    subprocess.run(['unzip', '-q', '-o', str(ARCHIVE_PATH), '-d', str(LOCAL_DATA)], check=True)
    assert IMAGES_ROOT.exists(), 'В архиве не найдена верхняя папка images/'
    EXTRACTED_MARKER.touch()
print('Изображения готовы:', IMAGES_ROOT)

In [ ]:
from huggingface_hub import snapshot_download
MODEL_ROOT = Path('/content/shared_models')
MODEL_DIR = MODEL_ROOT / 'PaddlePaddle' / 'PaddleOCR-VL-1.5'
snapshot_download(
    repo_id='PaddlePaddle/PaddleOCR-VL-1.5',
    local_dir=MODEL_DIR,
)
print('Модель готова:', MODEL_DIR)

In [ ]:
import os
import time

def run_ocr(limit, output_path, report_path):
    command = [
        sys.executable, '-m', 'ecup.features.ocr',
        '--manifest', str(MANIFEST_PATH),
        '--images-root', str(IMAGES_ROOT),
        '--cache-dir', str(CACHE_DIR),
        '--output', str(output_path),
        '--report', str(report_path),
        '--max-pixels', str(MAX_PIXELS),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--progress-every', '10',
    ]
    if limit is not None:
        command.extend(['--limit', str(limit)])
    environment = {
        **os.environ,
        'PYTHONPATH': str(REPO_DIR / 'src'),
        'SHARED_MODELS_PATH': str(MODEL_ROOT),
        'CUDA_VISIBLE_DEVICES': '0',
        'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    }
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO_DIR, env=environment, check=True)
    return time.perf_counter() - started

SMOKE_IMAGES = 10
seconds = run_ocr(
    SMOKE_IMAGES,
    OUTPUT_DIR / 'ocr_text_smoke.parquet',
    OUTPUT_DIR / 'R11-ocr-smoke.md',
)
seconds_per_image = seconds / SMOKE_IMAGES
estimated_days = seconds_per_image * 49_456 / 86_400
print(f'{seconds_per_image:.2f} s/image; full estimate: {estimated_days:.2f} days')

## Полный запуск

Запускайте эту ячейку только после проверки текста и скорости smoke-теста. При отключении Colab запустите notebook повторно: уже готовые изображения будут взяты из кеша Google Drive.

In [ ]:
run_ocr(
    None,
    OUTPUT_DIR / 'ocr_text.parquet',
    OUTPUT_DIR / 'R11-ocr-cache.md',
)